# Archived BB84 quick control

This early single-setting detector check uses a strict cutoff at 50 retained checks. The active terminal sweep reports the same detector comparison across settings and calibrates randomized ties. [Current BB84 notebook](../BB84.ipynb#terminal-sweeps)


In [ ]:
import numpy as np

STATE_ORDER = ("Z0", "Z1", "X+", "X-")
DETECTOR_NAMES = ("Pooled QBER", "Basis U2", "State U4")
Q_Z, Q_X = 0.002, 0.038
STATE_BASELINE = np.repeat([Q_Z, Q_X], 2)
BASELINE_TRIALS_PER_STATE = 1_000
RETAINED_CHECKS = 50
STUDY_TRIALS = 10_000
TARGET_FALSE_ALARM = 0.05

# Settings shared by the terminal and cumulative BB84 sweeps.
EVE_Z_PROBABILITIES = (0.1, 0.5, 0.9)
CHECK_BUDGETS = (20, 50, 100)
LOOK_FRACTIONS = (0.25, 0.5, 1.0)


In [ ]:
def sample_bb84_checks(
    rng, intercept_fraction, eve_p_z, *,
    retained_checks=RETAINED_CHECKS, trials=STUDY_TRIALS, state_baseline=STATE_BASELINE
):
    """Sample state totals and errors for fixed-size sifted BB84 trials."""
    counts = rng.multinomial(retained_checks, [0.25] * len(STATE_ORDER), size=trials)
    attack_error = 0.5 * intercept_fraction * np.repeat([1 - eve_p_z, eve_p_z], 2)
    error_rate = state_baseline + (1 - 2 * state_baseline) * attack_error
    return counts, rng.binomial(counts, error_rate)


def conditioned_score(counts, errors, baseline):
    variable = (baseline > 0) & (baseline < 1)
    rates = baseline[variable]
    weights = (1 - 2 * rates) / (rates * (1 - rates))
    score = ((errors[:, variable] - counts[:, variable] * rates) * weights).sum(axis=1)

    # A true rate of 0 or 1 makes any deviation conclusive evidence against the null.
    expected = counts[:, ~variable] * baseline[~variable]
    score[(errors[:, ~variable] != expected).any(axis=1)] = np.inf
    return score


def detector_scores(counts, errors, fitted_baseline):
    """Return pooled QBER, basis-grouped U2, and state-conditioned U4."""
    basis_baseline = fitted_baseline.reshape(2, 2).mean(axis=1)
    basis_counts = counts.reshape(-1, 2, 2).sum(axis=2)
    basis_errors = errors.reshape(-1, 2, 2).sum(axis=2)
    basis_score = conditioned_score(basis_counts, basis_errors, basis_baseline)
    if np.array_equal(fitted_baseline[::2], fitted_baseline[1::2]):
        # With equal rates within each basis, the two scores are algebraically identical.
        state_score = basis_score
    else:
        state_score = conditioned_score(counts, errors, fitted_baseline)
    return np.column_stack([errors.sum(axis=1) / counts.sum(axis=1), basis_score, state_score])

In [ ]:
fit_rng, cutoff_rng, null_rng, attack_rng = [
    np.random.default_rng(seed) for seed in np.random.SeedSequence(20261030).spawn(4)
]
calibration_errors = fit_rng.binomial(BASELINE_TRIALS_PER_STATE, STATE_BASELINE)
fitted_baseline = (calibration_errors + 0.5) / (BASELINE_TRIALS_PER_STATE + 1)
cutoff_checks = sample_bb84_checks(cutoff_rng, 0.0, 0.5)
cutoff_scores = detector_scores(*cutoff_checks, fitted_baseline)
cutoffs = np.quantile(cutoff_scores, 1 - TARGET_FALSE_ALARM, axis=0, method="higher")

null_checks = sample_bb84_checks(null_rng, 0.0, 0.5)
attack_checks = sample_bb84_checks(attack_rng, 0.1, 0.5)
# A strict comparison handles ties conservatively at the cutoff.
false_alarm_rates = (detector_scores(*null_checks, fitted_baseline) > cutoffs).mean(axis=0)
detection_power = (detector_scores(*attack_checks, fitted_baseline) > cutoffs).mean(axis=0)

print("Detector       False alarm    Power at f = 0.10")
for name, false_alarm, power in zip(
    DETECTOR_NAMES, false_alarm_rates, detection_power
):
    print(f"{name:<14} {false_alarm:>10.2%} {power:>19.2%}")